# Lab 04 · Batch, and the join that corrupts your data
**~15 minutes of work, then a wait · costs about $0.01 · Domain 2 API Mechanics**

50% off, no streaming, 24-hour window, and **results come back unordered**.
That last one is the exam question, and it is a silent data-corruption bug: the
batch succeeds, every row returns, and the labels are attached to the wrong
records.

In [ ]:
import os, anthropic
client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY
MODEL  = "claude-sonnet-4-6"            # verify against lab 00 output
CHEAP  = "claude-haiku-4-5"             # for high-volume steps
print("sdk", anthropic.__version__)

## Submit with custom_id

In [ ]:
REVIEWS = [
    "Arrived broken and support never replied.",
    "Exactly what I needed, shipped fast.",
    "Fine, nothing special.",
    "Second time this has failed. Done with this brand.",
    "Better than the previous model by a mile.",
]

batch = client.messages.batches.create(requests=[
    {"custom_id": f"review-{i}",
     "params": {"model": CHEAP, "max_tokens": 12,
                "messages": [{"role":"user",
                    "content": f"Reply with one word, positive or negative:\n{t}"}]}}
    for i, t in enumerate(REVIEWS)
])
print("batch:", batch.id, "|", batch.processing_status)

## Poll

Can take a few minutes. Batch is for work nobody is waiting on.

In [ ]:
import time
while True:
    b = client.messages.batches.retrieve(batch.id)
    print(b.processing_status, b.request_counts)
    if b.processing_status == "ended":
        break
    time.sleep(20)

## The bug, demonstrated

`positional` is what a reasonable engineer writes on a Friday. Compare it with
the `custom_id` join and see whether they agree.

In [ ]:
raw = list(client.messages.batches.results(batch.id))

positional = {REVIEWS[i]: r.result.message.content[0].text.strip()
              for i, r in enumerate(raw) if r.result.type == "succeeded"}

by_id = {}
for r in raw:
    if r.result.type == "succeeded":
        idx = int(r.custom_id.split("-")[1])
        by_id[REVIEWS[idx]] = r.result.message.content[0].text.strip()

print("returned order:", [r.custom_id for r in raw])
print()
for text in REVIEWS:
    p, c = positional.get(text), by_id.get(text)
    flag = "  <-- MISMATCH" if p != c else ""
    print(f"{text[:44]:<46} positional={p:<10} custom_id={c}{flag}")

If the returned order happened to match this time, run it again with more
requests. **The API makes no ordering guarantee**, so code that relies on order
is broken whether or not it has failed yet. That is the worst kind of bug.

---
### Checkpoint
- Four properties of the Batch API
- What is the only safe join key?
- Does batch stack with caching? At what discount?